# Iterators and Generators

Understanding lazy evaluation helps you work with large datasets efficiently.

## Iteration Protocol

Python's `for` loop uses two methods:
- `__iter__()` - returns an iterator object
- `__next__()` - returns next value or raises `StopIteration`

In [ ]:
# What happens behind the scenes in a for loop
numbers = [1, 2, 3]

# This:
for n in numbers:
    print(n, end=' ')
print()

# Is equivalent to:
iterator = iter(numbers)  # Calls __iter__()
while True:
    try:
        n = next(iterator)  # Calls __next__()
        print(n, end=' ')
    except StopIteration:
        break
print()

In [ ]:
# Creating an iterator class
class Countdown:
    def __init__(self, start):
        self.current = start
    
    def __iter__(self):
        return self
    
    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        self.current -= 1
        return self.current + 1

for n in Countdown(5):
    print(n, end=' ')
print()

## Generators - The Easy Way

In [ ]:
# Generator function - uses 'yield'
def countdown(n):
    while n > 0:
        yield n  # Pauses here, returns value
        n -= 1

# Much simpler than the iterator class!
for n in countdown(5):
    print(n, end=' ')
print()

In [ ]:
# Generator expression - like list comprehension but lazy
# List comprehension - creates all items immediately
squares_list = [x**2 for x in range(1000000)]

# Generator expression - creates items on demand
squares_gen = (x**2 for x in range(1000000))

print(f"List type: {type(squares_list)}")
print(f"Generator type: {type(squares_gen)}")

# Generator uses almost no memory until you iterate
print(f"First 5: {[next(squares_gen) for _ in range(5)]}")

## Why Generators Matter

In [ ]:
# Memory efficient - process large files line by line
def read_large_file(filepath):
    """Read file line by line without loading all into memory."""
    # with open(filepath) as f:
    #     for line in f:  # File objects are iterators!
    #         yield line.strip()
    # Simulated for demo:
    for i in range(5):
        yield f"Line {i}"

for line in read_large_file("big_file.txt"):
    print(line)

In [ ]:
# Pipeline processing
def numbers():
    for i in range(10):
        yield i

def doubled(items):
    for item in items:
        yield item * 2

def filtered(items):
    for item in items:
        if item > 5:
            yield item

# Chain generators - each item flows through pipeline
pipeline = filtered(doubled(numbers()))
print(f"Result: {list(pipeline)}")

## Generator Features

In [ ]:
# yield from - delegate to another generator
def flatten(nested):
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)  # Recursively yield
        else:
            yield item

nested = [1, [2, 3, [4, 5]], 6, [7, 8]]
print(f"Flattened: {list(flatten(nested))}")

In [ ]:
# Generator with return value
def accumulator():
    total = 0
    while True:
        value = yield total
        if value is None:
            return total  # Final return
        total += value

acc = accumulator()
print(next(acc))      # Start generator, get 0
print(acc.send(10))   # Send value, get 10
print(acc.send(20))   # Send value, get 30
print(acc.send(5))    # Send value, get 35

## Built-in Iterator Tools

In [ ]:
# map, filter, zip are lazy (return iterators)
numbers = [1, 2, 3, 4, 5]

mapped = map(lambda x: x * 2, numbers)
filtered = filter(lambda x: x > 2, numbers)
zipped = zip([1, 2, 3], ['a', 'b', 'c'])

print(f"map: {type(mapped)}")
print(f"filter: {type(filtered)}")
print(f"zip: {type(zipped)}")

# Convert to list to see results
print(f"mapped: {list(map(lambda x: x * 2, numbers))}")
print(f"filtered: {list(filter(lambda x: x > 2, numbers))}")

In [ ]:
# itertools - powerful iterator utilities
from itertools import (
    count,      # Infinite counter
    cycle,      # Infinite cycle through items
    chain,      # Chain iterables together
    islice,     # Slice an iterator
    takewhile,  # Take while condition true
    dropwhile,  # Drop while condition true
    groupby,    # Group consecutive items
)

# count - infinite
from itertools import islice
print(f"count: {list(islice(count(10), 5))}")

# chain - combine iterators
print(f"chain: {list(chain([1, 2], [3, 4], [5, 6]))}")

# takewhile
print(f"takewhile: {list(takewhile(lambda x: x < 5, range(10)))}")

In [ ]:
# groupby example
from itertools import groupby

data = [
    {'name': 'Alice', 'dept': 'Engineering'},
    {'name': 'Bob', 'dept': 'Engineering'},
    {'name': 'Charlie', 'dept': 'Sales'},
    {'name': 'Diana', 'dept': 'Sales'},
]

# Must be sorted by key first!
for dept, group in groupby(data, key=lambda x: x['dept']):
    names = [p['name'] for p in group]
    print(f"{dept}: {names}")

## AI Code Patterns

In [ ]:
# Pattern 1: Data batch generator
def batch_generator(items, batch_size):
    """Yield items in batches."""
    batch = []
    for item in items:
        batch.append(item)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:  # Don't forget the last partial batch!
        yield batch

data = range(10)
for batch in batch_generator(data, 3):
    print(f"Batch: {batch}")

In [ ]:
# Pattern 2: Windowed iteration
from collections import deque

def sliding_window(items, size):
    """Generate sliding windows over items."""
    window = deque(maxlen=size)
    for item in items:
        window.append(item)
        if len(window) == size:
            yield tuple(window)

data = [1, 2, 3, 4, 5]
for window in sliding_window(data, 3):
    print(f"Window: {window}")

In [ ]:
# Pattern 3: Infinite sequence with early exit
def fibonacci():
    """Infinite Fibonacci generator."""
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

# Get Fibonacci numbers less than 100
from itertools import takewhile
fibs = list(takewhile(lambda x: x < 100, fibonacci()))
print(f"Fibonacci < 100: {fibs}")

In [ ]:
# Pattern 4: Progress tracking generator
def with_progress(items, total=None):
    """Wrap iterator with progress reporting."""
    if total is None:
        items = list(items)
        total = len(items)
    
    for i, item in enumerate(items, 1):
        yield item
        if i % 3 == 0 or i == total:
            print(f"Progress: {i}/{total} ({100*i/total:.0f}%)")

data = range(10)
results = [x * 2 for x in with_progress(data)]

## Summary

| Concept | Description |
|---------|-------------|
| Iterator | Object with `__iter__` and `__next__` |
| Generator function | Function with `yield` |
| Generator expression | `(x for x in items)` |
| `yield from` | Delegate to sub-generator |
| itertools | Powerful iterator utilities |

## Next Up

Descriptors and properties - attribute magic.

Continue to: [Descriptors & Properties](03-descriptors-properties.ipynb)